# Intermediate 12 — Integrating Authorization with LLMs, Agents & Guardrails

## Enterprise scenario

A claims assistant can:

- read and update claims;
- search enterprise knowledge;
- call an MCP claims server;
- ask a research sub-agent for help;
- request payments;
- pause for human approval on sensitive actions.

The model is deliberately treated as an **untrusted planner**. Every consequential action crosses a deterministic authorization boundary.


In [ ]:
from dataclasses import dataclass, field, asdict
from datetime import datetime, timedelta, timezone
from typing import Any, Literal
import hashlib, json, uuid, copy
import pandas as pd

NOW=datetime.now(timezone.utc)


## 1 — Trusted security context

In [ ]:
@dataclass(frozen=True)
class SecurityContext:
    principal_id: str
    tenant_id: str
    agent_id: str
    workload_id: str
    task_id: str
    delegation_id: str
    assurance_level: int = 2

ctx=SecurityContext(
    principal_id="user:alice",
    tenant_id="acme",
    agent_id="claims-agent",
    workload_id="spiffe://corp.example/prod/claims-agent",
    task_id="task:483",
    delegation_id="del:483"
)
ctx


## 2 — Structured model intent

In [ ]:
@dataclass
class Intent:
    action: str
    resource: str
    tool: str
    purpose: str
    parameters: dict[str,Any]=field(default_factory=dict)

intent=Intent(
    action="claim.update",
    resource="claim:483",
    tool="claims.update",
    purpose="update claim status",
    parameters={"status":"reviewed"}
)
intent


## 3 — Never let the model supply trusted identity

In [ ]:
model_output={
    "principal_id":"user:ceo",
    "tenant_id":"other",
    "action":"claim.update",
    "resource":"claim:483"
}
print("Ignore model identity fields:", {k:model_output[k] for k in ["principal_id","tenant_id"]})
print("Trusted principal:",ctx.principal_id,"trusted tenant:",ctx.tenant_id)


## 4 — Normalize tool operations

In [ ]:
ACTION_MAP={
    "claims.get":"claim.read",
    "claims.update":"claim.update",
    "knowledge.search":"knowledge.search",
    "payments.create":"payment.create"
}
def normalize(tool):
    if tool not in ACTION_MAP:
        raise ValueError("unknown tool")
    return ACTION_MAP[tool]

normalize("claims.update")


## 5 — Delegation

In [ ]:
delegation={
    "id":"del:483",
    "delegatee":"claims-agent",
    "tenant":"acme",
    "actions":{"claim.read","claim.update","knowledge.search"},
    "resources":{"claim:483","kb:claims"},
    "expires_at":NOW+timedelta(hours=1),
    "active":True
}


## 6 — Decision contract

In [ ]:
@dataclass
class Decision:
    outcome: Literal["allow","deny","step_up"]
    decision_id: str
    reason: str
    constraints: dict[str,Any]=field(default_factory=dict)
    obligations: list[str]=field(default_factory=list)
    expires_at: datetime|None=None


## 7 — Simulated PDP

In [ ]:
def authorize(ctx:SecurityContext,intent:Intent,delegation:dict)->Decision:
    if ctx.tenant_id != delegation["tenant"]:
        return Decision("deny",str(uuid.uuid4()),"TENANT_MISMATCH")
    if ctx.agent_id != delegation["delegatee"]:
        return Decision("deny",str(uuid.uuid4()),"WRONG_DELEGATEE")
    if not delegation["active"] or delegation["expires_at"] <= NOW:
        return Decision("deny",str(uuid.uuid4()),"DELEGATION_INACTIVE")
    if intent.action not in delegation["actions"]:
        if intent.action=="payment.create":
            return Decision("step_up",str(uuid.uuid4()),"PAYMENT_AUTHORITY_REQUIRED")
        return Decision("deny",str(uuid.uuid4()),"ACTION_OUT_OF_SCOPE")
    if intent.resource not in delegation["resources"]:
        return Decision("deny",str(uuid.uuid4()),"RESOURCE_OUT_OF_SCOPE")
    constraints={}
    if intent.action=="claim.update":
        constraints["allowed_fields"]={"status","notes"}
    return Decision(
        "allow",str(uuid.uuid4()),"TASK_SCOPE",
        constraints=constraints,
        obligations=["audit"],
        expires_at=NOW+timedelta(seconds=60)
    )

decision=authorize(ctx,intent,delegation)
decision


## 8 — Enforce constraints, not only ALLOW

In [ ]:
def enforce_constraints(intent,decision):
    if decision.outcome!="allow":
        raise PermissionError(decision.reason)
    allowed_fields=decision.constraints.get("allowed_fields")
    if allowed_fields is not None:
        attempted=set(intent.parameters)
        if not attempted.issubset(allowed_fields):
            raise PermissionError("FIELD_CONSTRAINT_VIOLATION")
    return True

enforce_constraints(intent,decision)


## 9 — Attack: model adds unauthorized field

In [ ]:
tampered=copy.deepcopy(intent)
tampered.parameters={"status":"reviewed","payout":100000}
try:
    enforce_constraints(tampered,decision)
except Exception as e:
    print("blocked:",e)


## 10 — Tool exposure filtering

In [ ]:
TOOLS={
 "claims.get":{"action":"claim.read"},
 "claims.update":{"action":"claim.update"},
 "claims.delete":{"action":"claim.delete"},
 "payments.create":{"action":"payment.create"},
 "knowledge.search":{"action":"knowledge.search"}
}
visible=[
 name for name,spec in TOOLS.items()
 if spec["action"] in delegation["actions"]
]
visible


## 11 — Tool visibility is not authorization

In [ ]:
direct_request=Intent(
    "claim.delete","claim:483","claims.delete","model tried hidden tool",{}
)
authorize(ctx,direct_request,delegation)


## 12 — Resource-level authorization

In [ ]:
other_claim=Intent("claim.read","claim:999","claims.get","read",{})
authorize(ctx,other_claim,delegation)


## 13 — Guardrail vs authorization

In [ ]:
def tool_guardrail(intent):
    # Content/business validation, not identity authority.
    if "payout" in intent.parameters and intent.parameters["payout"] > 10000:
        return False,"HIGH_VALUE_ARGUMENT"
    return True,"VALID"

print("guardrail:",tool_guardrail(intent))
print("authorization:",authorize(ctx,intent,delegation).outcome)


## 14 — Risk scoring

In [ ]:
def risk(intent):
    if intent.action=="payment.create":
        amount=float(intent.parameters.get("amount",0))
        if amount>10000: return "critical"
        if amount>500: return "high"
        return "medium"
    if intent.action in {"claim.delete","claim.export"}:
        return "high"
    return "low"

risk(intent)


## 15 — Risk-based approval

In [ ]:
def approval_required(intent):
    return risk(intent) in {"high","critical"}

payment=Intent(
    "payment.create","account:42","payments.create",
    "settle approved claim",{"amount":750,"currency":"CAD"}
)
approval_required(payment)


## 16 — Bind approval to the transaction

In [ ]:
def transaction_digest(ctx,intent):
    payload={
      "principal":ctx.principal_id,
      "agent":ctx.agent_id,
      "task":ctx.task_id,
      "action":intent.action,
      "resource":intent.resource,
      "tool":intent.tool,
      "parameters":intent.parameters
    }
    return hashlib.sha256(json.dumps(
        payload,sort_keys=True,separators=(",",":")
    ).encode()).hexdigest()

approved_digest=transaction_digest(ctx,payment)
approved_digest[:20]


## 17 — Parameter change invalidates approval

In [ ]:
changed=copy.deepcopy(payment)
changed.parameters["amount"]=7500
print("approval still matches?",approved_digest==transaction_digest(ctx,changed))


## 18 — Approval record

In [ ]:
approval={
 "approval_id":"apr:1",
 "approver":"user:manager",
 "transaction_digest":approved_digest,
 "approved_at":NOW,
 "expires_at":NOW+timedelta(minutes=5)
}


## 19 — Approval freshness

In [ ]:
def valid_approval(ctx,intent,approval,now=NOW):
    return (
      approval["expires_at"] > now and
      approval["transaction_digest"] == transaction_digest(ctx,intent)
    )

valid_approval(ctx,payment,approval)


## 20 — Delayed approval must be revalidated

In [ ]:
future=NOW+timedelta(minutes=10)
print("valid after delay?",valid_approval(ctx,payment,approval,future))


## 21 — Step-up

In [ ]:
payment_decision=authorize(ctx,payment,delegation)
payment_decision


A `step_up` result should lead to a controlled flow that obtains additional authority/approval and then re-runs authorization. Do not mutate `step_up` into `allow` inside the LLM loop.

## 22 — Safe replanning after denial

In [ ]:
def safe_options(decision):
    if decision.reason=="ACTION_OUT_OF_SCOPE":
        return ["claim.read","request_approval"]
    if decision.reason=="RESOURCE_OUT_OF_SCOPE":
        return ["request_resource_access","stop"]
    return ["stop"]

safe_options(authorize(ctx,direct_request,delegation))


## 23 — Authorization-aware RAG

In [ ]:
documents=[
 {"id":"doc:1","tenant":"acme","groups":{"claims"},"text":"ACME claims guide"},
 {"id":"doc:2","tenant":"other","groups":{"claims"},"text":"Other tenant confidential"},
 {"id":"doc:3","tenant":"acme","groups":{"hr"},"text":"ACME HR confidential"},
]
user_groups={"claims"}
authorized_docs=[
 d for d in documents
 if d["tenant"]==ctx.tenant_id and bool(d["groups"] & user_groups)
]
authorized_docs


## 24 — Memory is a resource

In [ ]:
memories=[
 {"id":"mem:1","tenant":"acme","owner":"user:alice","text":"claim preference"},
 {"id":"mem:2","tenant":"acme","owner":"user:bob","text":"private note"},
]
[m for m in memories if m["tenant"]==ctx.tenant_id and m["owner"]==ctx.principal_id]


## 25 — Multi-agent attenuation

In [ ]:
parent_actions={"claim.read","claim.update","knowledge.search"}
research_required={"knowledge.search"}
child_actions=parent_actions & research_required
print(child_actions)
assert child_actions.issubset(parent_actions)


## 26 — Handoff authorization

In [ ]:
HANDOFFS={
 ("claims-agent","research-agent"):{"knowledge.search"},
 ("claims-agent","payment-agent"):{"payment.request"}
}
def authorize_handoff(source,target,capability):
    return capability in HANDOFFS.get((source,target),set())

print(authorize_handoff("claims-agent","research-agent","knowledge.search"))
print(authorize_handoff("claims-agent","payment-agent","payment.execute"))


## 27 — MCP server identity and tool authorization

In [ ]:
def authorize_mcp(server_id,tool,allowed_server,allowed_tools):
    if server_id!=allowed_server:
        return False,"UNTRUSTED_MCP_SERVER"
    if tool not in allowed_tools:
        return False,"MCP_TOOL_DENIED"
    return True,"ALLOW"

authorize_mcp("mcp:claims-prod","claim.read","mcp:claims-prod",{"claim.read"})


## 28 — MCP token audience/resource binding

In [ ]:
token={"aud":"https://mcp.claims.example","scope":"claims.read"}
expected_resource="https://mcp.claims.example"
print("audience valid?",token["aud"]==expected_resource)


## 29 — Reject token passthrough

In [ ]:
incoming_token={"aud":"https://mcp.claims.example","value":"client-token"}
downstream_api="https://claims-api.internal"
print("May forward same token downstream?",incoming_token["aud"]==downstream_api)
print("Use a separate downstream token intended for the downstream API.")


## 30 — MCP task authorization

In [ ]:
tasks=[
 {"id":"task-mcp-1","owner":"user:alice","tenant":"acme","result":"..."},
 {"id":"task-mcp-2","owner":"user:bob","tenant":"acme","result":"..."},
]
[t for t in tasks if t["owner"]==ctx.principal_id and t["tenant"]==ctx.tenant_id]


## 31 — Policy-engine input

In [ ]:
policy_input={
 "principal":{"id":ctx.principal_id,"authenticated":True,"tenant":ctx.tenant_id},
 "agent":{"id":ctx.agent_id},
 "workload":{"id":ctx.workload_id,"approved":True},
 "delegation":{
   "id":delegation["id"],"active":delegation["active"],
   "delegatee":delegation["delegatee"],
   "actions":sorted(delegation["actions"]),
   "resources":sorted(delegation["resources"])
 },
 "action":intent.action,
 "resource":{"id":intent.resource,"tenant":ctx.tenant_id},
 "parameters":intent.parameters,
 "risk":{"level":risk(intent)}
}
print(json.dumps(policy_input,indent=2))


## 32 — OPA exercise

The repository includes:

```text
policies/opa/agent_tools.rego
```

Replace the simulated `authorize()` function with a call to OPA and verify that the tool dispatcher refuses to execute on `deny`.

Then add policies for:

```text
high-risk approval
field-level constraints
cross-tenant denial
workload assurance
delegation expiry
```


## 33 — Cedar exercise

The repository includes:

```text
policies/cedar/agent_tools.cedar
```

Map:

```text
principal -> Agent entity
action    -> normalized tool/data operation
resource  -> Claim/Tool/Data object
context   -> workload, task, delegation, risk, approval
```

Test default-deny and forbid semantics.


## 34 — OpenFGA exercise

The repository includes:

```text
policies/openfga/model.fga
```

Extend it with relationships such as:

```text
agent acts_for user
agent assigned_to task
agent can_invoke tool
user member_of tenant
claim belongs_to tenant
```

Use contextual policy separately for volatile risk/assurance facts.


## 35 — LangGraph secure graph pattern

The repository includes:

```text
langgraph/secure_agent_graph.py
```

Production graph:

```text
planner
  ↓
normalize
  ↓
authorize
  ├── deny → safe replan
  ├── step_up → interrupt/HITL
  └── allow → execute
                    ↓
                 evidence
```

The authorization node must not rely on model-generated identity claims.


## 36 — OpenAI Agents SDK function-tool pattern

Current Agents SDK patterns support function tools, guardrails, HITL, MCP integration, sessions, and tracing.

A production function tool can place the PEP immediately before the real side effect:

```python
@function_tool
async def update_claim(ctx, claim_id: str, status: str):
    security = ctx.context.security
    intent = Intent(...)

    decision = await pdp.authorize(security, intent)

    if decision.outcome != "allow":
        return safe_denial(decision)

    enforce_constraints(decision, intent)
    return await claims_api.update(...)
```

Keep the API/resource authorization as defense in depth.


## 37 — OpenAI Agents SDK HITL pattern

The current SDK can pause runs when a tool requires approval and resume from `RunState`.

Security design:

```text
tool requested
  ↓
validate arguments
  ↓
authorize
  ↓
approval required?
  ↓
pause
  ↓
authorized human approves exact transaction
  ↓
resume
  ↓
revalidate approval + authorization
  ↓
execute
```

Do not interpret arbitrary conversational text such as "yes" as equivalent to a cryptographically/structurally bound approval record.


## 38 — OpenAI Agents SDK MCP pattern

Current SDK MCP integrations support approval policies and tool filtering. Local MCP integrations also support per-call metadata resolution.

Use those features to reduce exposure and propagate correlation/business context, while still requiring the MCP server to validate its own authorization context.

Never treat a model-visible MCP tool description as trusted security policy.


## 39 — Evidence event

In [ ]:
def evidence(ctx,intent,decision,result):
    return {
      "trace_id":uuid.uuid4().hex,
      "decision_id":decision.decision_id,
      "principal_id":ctx.principal_id,
      "agent_id":ctx.agent_id,
      "workload_id":ctx.workload_id,
      "task_id":ctx.task_id,
      "delegation_id":ctx.delegation_id,
      "action":intent.action,
      "resource":intent.resource,
      "tool":intent.tool,
      "decision":decision.outcome,
      "reason":decision.reason,
      "result":result,
      "timestamp":NOW.isoformat()
    }

evidence(ctx,intent,decision,"success")


## 40 — Attack: prompt asks for admin tool

In [ ]:
malicious_model_intent=Intent(
 "admin.export_all","tenant:acme","admin.export",
 "prompt said to ignore previous rules",{}
)
authorize(ctx,malicious_model_intent,delegation)


## 41 — Attack: cross-tenant resource substitution

In [ ]:
cross_tenant_ctx=SecurityContext(
 ctx.principal_id,"other",ctx.agent_id,ctx.workload_id,ctx.task_id,ctx.delegation_id
)
authorize(cross_tenant_ctx,intent,delegation)


## 42 — Attack: stale approval

In [ ]:
print(valid_approval(ctx,payment,approval,NOW+timedelta(hours=1)))


## 43 — Attack: policy outage

In [ ]:
def pep_call(pdp_available,high_risk=True):
    if not pdp_available:
        return {"executed":False,"reason":"PDP_UNAVAILABLE"} if high_risk else {"executed":False,"reason":"DEGRADED_DENY"}
    return {"executed":True}

pep_call(False,True)


## 44 — Attack: direct tool bypass

In [ ]:
execution_paths=pd.DataFrame([
 {"path":"agent -> tool_router -> PDP -> API","authorized":True},
 {"path":"agent -> API directly","authorized":False},
 {"path":"agent -> MCP -> API","authorized":True},
])
execution_paths[~execution_paths.authorized]


## 45 — Secure execution function

In [ ]:
def secure_execute(ctx,intent,delegation):
    # 1. Validate/guardrail
    ok,msg=tool_guardrail(intent)
    if not ok:
        return {"executed":False,"stage":"guardrail","reason":msg}

    # 2. Authorize
    d=authorize(ctx,intent,delegation)
    if d.outcome!="allow":
        return {"executed":False,"stage":"authorization","reason":d.reason,
                "decision_id":d.decision_id}

    # 3. Enforce constraints
    try:
        enforce_constraints(intent,d)
    except PermissionError as e:
        return {"executed":False,"stage":"constraints","reason":str(e)}

    # 4. Real tool would execute here.
    result={"status":"simulated-success"}

    # 5. Evidence
    return {
      "executed":True,
      "result":result,
      "evidence":evidence(ctx,intent,d,"success")
    }

secure_execute(ctx,intent,delegation)


## 46 — Capstone lab

Extend `secure_execute()` into a small enterprise agent runtime.

Requirements:

1. model output is parsed into a typed intent;
2. trusted identity is injected server-side;
3. tools are filtered by task authority;
4. every invocation receives resource-level authorization;
5. high-risk operations return step-up/HITL;
6. approvals bind to a transaction digest;
7. changed parameters invalidate approval;
8. sub-agent authority is attenuated;
9. MCP server identity and token audience are checked;
10. RAG results are tenant/ACL filtered;
11. memory is authorization-aware;
12. policy outages fail safely;
13. every decision/action produces evidence;
14. adversarial tests from Intermediate 11 are reused.

The learner should then replace the simulated PDP with **OPA, Cedar, or OpenFGA + contextual policy**, and replace the simulated planner with either **OpenAI Agents SDK** or a **LangGraph/LangChain** workflow.


# Review questions

1. Why should an LLM be treated as an untrusted planner?
2. Why is a system prompt not an authorization policy?
3. What is a structured intent?
4. Why normalize actions across tools/APIs?
5. What is the difference between PDP and PEP?
6. Why is a rich decision contract useful?
7. What are authorization constraints?
8. What are policy obligations?
9. How is tool filtering different from tool authorization?
10. Why do guardrails not replace authorization?
11. Why does authorization not replace guardrails?
12. Why should resource authorization happen below tool-level authorization?
13. What is approval binding?
14. Why should approval expire?
15. When should authorization be rerun?
16. What is step-up authorization?
17. Why should trusted context be kept separate from model output?
18. How should RAG be authorization-aware?
19. Why is memory an authorization surface?
20. How should authority change during agent handoffs?
21. What is authority attenuation?
22. Why is an MCP server a security boundary?
23. What are OAuth resource indicators used for in MCP?
24. Why is token passthrough dangerous?
25. Why should MCP scopes be minimized?
26. Why must MCP tasks be access controlled?
27. Where can authorization live in LangGraph?
28. Where should a function-tool PEP live?
29. Why revalidate after HITL?
30. What evidence should every consequential agent action produce?

# Next

## Intermediate 13 — Capstone: Secure Agent Identity & Authorization Architecture
